# M4 / M23 — Audio Spectrogram Transformer on ICBHI sound events

**Member A (Asif Mahbub)** · shared representation + AST backbone
**Supervisor:** Dr. Riasat Khan · **Target venue:** *Biomedical Signal Processing and Control*

---

### What this notebook produces

| Run | `USE_SPECAUGMENT` | Model ID | Role |
|---|---|---|---|
| 1 | `False` | **M4** | AST clean — backbone-ablation cell |
| 2 | `True` | **M23** | AST + SpecAugment — augmentation ablation (assignment item 5) |

Run the notebook end to end twice, flipping the flag in the config cell. Nothing
else changes, which is what makes the pair a valid ablation.

### And one thing beyond the assignment

Section 9 measures **open-world separability**: how well the frozen AST
representation separates the 19 held-out unseen-disease patients from known ones,
*without ever training on a disease label*. That is the M12 selection criterion,
and it is the part of this workstream that is a contribution rather than a
benchmark re-run.

### Task scope

4-class sound-event classification (Normal / Crackle / Wheeze / Both) at cycle
level. **Not** disease diagnosis — 6,898 cycles and four clean classes, with no
patient-level aggregation to explain in a six-minute video. Disease diagnosis has
126 patients, a 64/126 COPD imbalance, and an unresolved device-shortcut question
(see `scripts/shortcut_probe.py`); it is Member B's task and it comes later.

---
## 0. Runtime setup

Mounts Drive, clones the project repo (for the shared `owmtl` package), and
installs what Colab lacks. Safe to re-run.

In [ ]:
# --- 0.1 Drive ---------------------------------------------------------------
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DRIVE_ROOT = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/OWMTL"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive root:", DRIVE_ROOT)

In [ ]:
# --- 0.2 Project repo (gives us the shared owmtl package) --------------------
REPO_URL  = "https://github.com/barshon-basak/CSE465-Project-OWMTL.git"
REPO_DIR  = "/content/CSE465-Project-OWMTL"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

# The package lives in the "Asif's" folder; note the apostrophe.
PKG_PARENT = os.path.join(REPO_DIR, "Asif's")
if PKG_PARENT not in sys.path:
    sys.path.insert(0, PKG_PARENT)

from owmtl import icbhi, features, reporting, separability
print("owmtl package loaded from", PKG_PARENT)

In [ ]:
# --- 0.3 Dependencies --------------------------------------------------------
def ensure(pkgs):
    missing = []
    for mod, pip_name in pkgs:
        try:
            __import__(mod)
        except ImportError:
            missing.append(pip_name)
    if missing:
        print("installing:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure([("torch", "torch"), ("torchaudio", "torchaudio"), ("transformers", "transformers"),
        ("librosa", "librosa"), ("soundfile", "soundfile"), ("sklearn", "scikit-learn"),
        ("matplotlib", "matplotlib"), ("tqdm", "tqdm")])

import torch, numpy as np
print("torch      :", torch.__version__)
print("cuda       :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

---
## 1. Dataset

The ICBHI 2017 corpus via the vbookshelf Kaggle mirror. Upload your
`kaggle.json` (Kaggle → Settings → API → Create New Token) when prompted.

> The mirror **omits** `ICBHI_challenge_train_test.txt`. Without it the official
> 60/40 challenge split is unavailable and our numbers cannot be placed next to
> published ICBHI results. If you can obtain that file, drop it in the dataset
> root before running Section 2 — everything downstream picks it up automatically.

In [ ]:
DATA_ROOT = "/content/Respiratory_Sound_Database"

if not os.path.isdir(DATA_ROOT):
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        from google.colab import files
        print("Upload kaggle.json ...")
        up = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        with open("/root/.kaggle/kaggle.json", "wb") as fh:
            fh.write(next(iter(up.values())))
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    ensure([("kaggle", "kaggle")])
    subprocess.run(["kaggle", "datasets", "download", "-d",
                    "vbookshelf/respiratory-sound-database", "-p", "/content",
                    "--unzip"], check=True)

# The mirror unzips to a doubly nested path; icbhi.locate() handles either shape.
for cand in ["/content/Respiratory_Sound_Database",
             "/content/respiratory-sound-database/Respiratory_Sound_Database"]:
    if os.path.isdir(cand):
        DATA_ROOT = cand
        break

paths = icbhi.locate(DATA_ROOT)
print("audio_dir           :", paths.audio_dir)
print("diagnosis_file      :", paths.diagnosis_file)
print("official_split_file :", paths.official_split_file or "NOT FOUND")

---
## 2. The shared split

Both artifacts are committed in the repo, so **load them** — do not rebuild.
Every member's results are comparable only because all four read these exact
files. The rebuild path below exists for regenerating `v2` deliberately.

Two things about this split are non-standard and both are deliberate:

1. **The 19 unseen-disease patients** (Bronchiectasis 7, Pneumonia 6,
   Bronchiolitis 6) are `held_out` — excluded from *every* training partition for
   *every* task. Under the official ICBHI split most of them fall in the training
   half, which would mean the shared backbone had already seen the audio of the
   patients later presented to it as "unknown". Member B's open-world result would
   be partly memorisation.
2. **A `calib` partition** is carved from the known classes and never trained on.
   Member D fits temperature scaling and the conformal wrapper there.

In [ ]:
SPLIT_DIR  = os.path.join(PKG_PARENT, "splits")
INDEX_PATH = os.path.join(SPLIT_DIR, "cycles_index_v1.csv")
SPLIT_PATH = os.path.join(SPLIT_DIR, "split_v1.json")

REBUILD = not (os.path.exists(INDEX_PATH) and os.path.exists(SPLIT_PATH))
if REBUILD:
    print("Committed split not found — building it now.\n")
    rows = icbhi.build_cycle_index(paths)
    official = (icbhi.read_official_split(paths.official_split_file)
                if paths.official_split_file else None)
    split_obj = icbhi.build_split(rows, seed=42, test_frac=0.40, calib_frac=0.15,
                                  official_by_stem=official)
    payload = icbhi.split_to_dict(split_obj, rows, paths)
    icbhi.write_cycle_index(rows, INDEX_PATH)
    icbhi.write_split(payload, SPLIT_PATH)

INDEX = icbhi.read_cycle_index(INDEX_PATH)
SPLIT = icbhi.load_split(SPLIT_PATH)

print(icbhi.summarise(SPLIT))
problems = icbhi.validate(SPLIT, INDEX)
print("\nvalidation:", "PASSED" if not problems else problems)
assert not problems, "split failed validation — do not train on it"

In [ ]:
# Which scheme are we reporting on?
#   "owmtl"    leak-free, required for anything feeding the open-world claim
#   "official" ICBHI 60/40, for literature comparability (needs the split file)
SCHEME = "official" if SPLIT["partitions"]["official"]["available"] else "owmtl"
print("reporting scheme:", SCHEME)

if SCHEME == "owmtl":
    train_rows = icbhi.select_cycles(INDEX, SPLIT, partition="train")
    val_rows   = icbhi.select_cycles(INDEX, SPLIT, partition="calib")
    test_rows  = icbhi.select_cycles(INDEX, SPLIT, partition="test")
else:
    train_rows = icbhi.select_cycles(INDEX, SPLIT, partition="train", scheme="official")
    test_rows  = icbhi.select_cycles(INDEX, SPLIT, partition="test",  scheme="official")
    # Carve a patient-independent validation slice out of official train.
    val_pids = set(SPLIT["partitions"]["owmtl"]["sound_event"]["calib"])
    val_rows   = [r for r in train_rows if r["patient_id"] in val_pids]
    train_rows = [r for r in train_rows if r["patient_id"] not in val_pids]

# The 19 unseen-disease patients — scored in Section 9, never trained on.
UNKNOWN_PIDS = set(icbhi.disease_patients(SPLIT, "unknown_eval"))
held_rows = [r for r in INDEX if r["patient_id"] in UNKNOWN_PIDS]

from collections import Counter
for name, rs in [("train", train_rows), ("val", val_rows),
                 ("test", test_rows), ("held_out(unknown)", held_rows)]:
    dist = dict(Counter(r["label_name"] for r in rs))
    n_pat = len(set(r["patient_id"] for r in rs))
    print("{:20s} {:6d} cycles  {:3d} patients  {}".format(name, len(rs), n_pat, dist))

assert not ({r["patient_id"] for r in train_rows} & {r["patient_id"] for r in test_rows})
assert not (UNKNOWN_PIDS & {r["patient_id"] for r in train_rows}), "unknown-disease leak"

---
## 3. Preprocessing — the seven stages

**This is a graded item.** The assignment marks *"apply ALL the preprocessing
techniques"* separately, so the stages are enumerated explicitly here and echoed
into the results JSON. Put this list on a slide; do not leave it implicit in code.

In [ ]:
CFG_AUDIO = features.AudioConfig()   # 16 kHz · 8 s · 128 mel · n_fft 1024 · hop 160 · 50-2000 Hz

print("PREPROCESSING PIPELINE")
for i, stage in enumerate(features.PREPROCESSING_STAGES, 1):
    print(f"  {i}. {stage}")
print("\n  8. SpecAugment  (training-time only — augmentation, not preprocessing;"
      "\n                   reported as the separate M4 -> M23 ablation)")
print()
for k, v in CFG_AUDIO.to_dict().items():
    if k != "preprocessing_stages":
        print(f"  {k:16s} {v}")

In [ ]:
# --- 3.1 Visualise the pipeline on one cycle of each class (slide figure) -----
import librosa, matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 4, figsize=(17, 12))
col_titles = ["1-2. raw cycle", "3-5. filtered + normalised",
              "6-7. log-mel (shared)", "AST fbank front-end"]

for row, cls in enumerate(icbhi.SOUND_EVENT_CLASSES):
    r = next(x for x in train_rows if x["label_name"] == cls)
    wav_path = os.path.join(paths.audio_dir, r["stem"] + ".wav")
    raw, _ = librosa.load(wav_path, sr=CFG_AUDIO.sample_rate, mono=True,
                          offset=r["start"], duration=r["end"] - r["start"])
    clean = features.load_cycle(wav_path, r["start"], r["end"], CFG_AUDIO)
    lm    = features.log_mel(clean, CFG_AUDIO)
    fb    = features.ast_fbank(clean, CFG_AUDIO)

    axes[row, 0].plot(np.linspace(0, len(raw) / CFG_AUDIO.sample_rate, len(raw)), raw, lw=0.4)
    axes[row, 1].plot(np.linspace(0, CFG_AUDIO.duration_s, len(clean)), clean, lw=0.4, color="#2b6cb0")
    axes[row, 2].imshow(lm, aspect="auto", origin="lower", cmap="magma")
    axes[row, 3].imshow(fb.T, aspect="auto", origin="lower", cmap="magma")
    axes[row, 0].set_ylabel(cls, fontsize=12, fontweight="bold")
    for c in range(4):
        axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
        if row == 0:
            axes[row, c].set_title(col_titles[c], fontsize=11)

fig.suptitle("Preprocessing pipeline, one cycle per class", fontsize=13)
fig.tight_layout()
os.makedirs("/content/figs", exist_ok=True)
fig.savefig("/content/figs/preprocessing_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Run configuration

**Flip `USE_SPECAUGMENT` here and re-run the whole notebook for the second run.**

In [ ]:
# ==================== THE ONLY THING THAT CHANGES BETWEEN RUNS ===============
USE_SPECAUGMENT = False        # False -> M4 (clean) | True -> M23 (+ SpecAugment)
# =============================================================================

MODEL_ID   = "M23" if USE_SPECAUGMENT else "M4"
MODEL_NAME = "AST (Audio Spectrogram Transformer)" + (" + SpecAugment" if USE_SPECAUGMENT else "")
MEMBER, MEMBER_NAME = "A", "Asif Mahbub"

AST_CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"
SEED           = 42

CFG = dict(
    architecture      = "AST_base_384_audioset_pretrained",
    frontend          = "kaldi_fbank",   # AST's own front-end — see features.py
    checkpoint        = AST_CHECKPOINT,
    sample_rate       = CFG_AUDIO.sample_rate,
    n_mels            = CFG_AUDIO.n_mels,
    target_frames     = 1024,
    batch_size        = 12,
    grad_accum_steps  = 2,               # effective batch 24
    num_epochs        = 25,
    lr_backbone       = 1e-5,            # AST fine-tuning is unstable above ~5e-5
    lr_head           = 1e-3,
    weight_decay      = 5e-7,
    optimizer         = "AdamW",
    scheduler         = "cosine_with_warmup",
    warmup_ratio      = 0.1,
    label_smoothing   = 0.0,
    class_weighted_loss = True,
    amp               = True,
    seed              = SEED,
    split_scheme      = SCHEME,
    primary_metric    = "icbhi_score",   # official challenge definition
)

CKPT_LOCAL = f"/content/ckpt/{MODEL_ID}"
CKPT_DRIVE = os.path.join(DRIVE_ROOT, MODEL_ID) if DRIVE_ROOT else CKPT_LOCAL
RESULT_DIR = os.path.join(CKPT_DRIVE, "results")
for d in (CKPT_LOCAL, CKPT_DRIVE, RESULT_DIR):
    os.makedirs(d, exist_ok=True)

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"{MODEL_ID}: {MODEL_NAME}")
print("checkpoints ->", CKPT_DRIVE)

---
## 5. Feature cache

Every cycle is decoded **once** into a memory-mapped array on Colab local disk
(~1.4 GB in float16). Recordings are grouped so each of the 920 `.wav` files is
read a single time rather than once per cycle.

Keep this off Drive — it regenerates in minutes and the I/O would be painful.

In [ ]:
CACHE_DIR = "/content/cache"
FRONTEND  = "ast_fbank"

if not os.path.exists(os.path.join(CACHE_DIR, f"{FRONTEND}.npy")):
    features.precompute_features(INDEX, paths.audio_dir, CACHE_DIR,
                                 frontend=FRONTEND, cfg=CFG_AUDIO, dtype="float16")

FEATS, FEAT_META = features.load_feature_cache(CACHE_DIR, FRONTEND)
UID_TO_ROW = {uid: i for i, uid in enumerate(FEAT_META["cycle_uids"])}
print("cache:", FEATS.shape, FEATS.dtype,
      f"{FEATS.nbytes / 1e9:.2f} GB")

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CycleDataset(Dataset):
    """Cycle-level sound-event dataset backed by the precomputed fbank cache.

    SpecAugment is applied here and only when `augment=True`, i.e. training only —
    the val and test loaders never see it, which is what makes M4 vs M23 a clean
    ablation rather than two differently-evaluated models.
    """

    def __init__(self, rows, feats, uid_to_row, *, augment=False, seed=SEED):
        self.rows = list(rows)
        self.feats = feats
        self.uid_to_row = uid_to_row
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        x = np.asarray(self.feats[self.uid_to_row[row["cycle_uid"]]], dtype=np.float32)
        if self.augment:
            # AST fbank is (frames, mels) -> frequency is axis 1.
            x = features.spec_augment(x, rng=self.rng, freq_axis=1)
        return torch.from_numpy(x), int(row["label"])


train_ds = CycleDataset(train_rows, FEATS, UID_TO_ROW, augment=USE_SPECAUGMENT)
val_ds   = CycleDataset(val_rows,   FEATS, UID_TO_ROW, augment=False)
test_ds  = CycleDataset(test_rows,  FEATS, UID_TO_ROW, augment=False)

loader = lambda ds, shuffle: DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                                        num_workers=2, pin_memory=True, drop_last=False)
train_loader, val_loader, test_loader = loader(train_ds, True), loader(val_ds, False), loader(test_ds, False)

xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), "labels:", tuple(yb.shape),
      "| augment:", USE_SPECAUGMENT)

---
## 6. Model

`ASTModel` (86.6 M params, ImageNet → AudioSet pretrained) plus a LayerNorm +
linear head, matching the head `ASTForAudioClassification` uses.

`forward(..., return_embedding=True)` returns the 768-d pooled representation
before the head. That embedding is the object Section 9 measures and the artifact
Members B/C/D actually consume — the head is disposable, the representation is
the deliverable.

In [ ]:
import torch.nn as nn
from transformers import ASTModel

class ASTSoundEvent(nn.Module):
    def __init__(self, checkpoint, n_classes=4, dropout=0.1):
        super().__init__()
        self.backbone = ASTModel.from_pretrained(checkpoint)
        d = self.backbone.config.hidden_size
        self.norm = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(d, n_classes)

    def forward(self, x, return_embedding=False):
        emb = self.backbone(input_values=x).pooler_output      # (B, 768)
        logits = self.head(self.dropout(self.norm(emb)))
        return (logits, emb) if return_embedding else logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ASTSoundEvent(AST_CHECKPOINT, n_classes=len(icbhi.SOUND_EVENT_CLASSES)).to(device)

params = reporting.count_params(model)
print(f"total params    : {params['total_params']:,}")
print(f"trainable params: {params['trainable_params']:,}")
print(f"model size      : {reporting.model_size_mb(model)} MB")

In [ ]:
from collections import Counter
from transformers import get_cosine_schedule_with_warmup

# Class weights from the TRAIN partition only.
counts = Counter(r["label"] for r in train_rows)
n_cls = len(icbhi.SOUND_EVENT_CLASSES)
weights = torch.tensor(
    [len(train_rows) / (n_cls * counts[i]) for i in range(n_cls)],
    dtype=torch.float32, device=device)
print("class weights:", {icbhi.SOUND_EVENT_CLASSES[i]: round(float(weights[i]), 3)
                         for i in range(n_cls)})

criterion = nn.CrossEntropyLoss(weight=weights if CFG["class_weighted_loss"] else None,
                                label_smoothing=CFG["label_smoothing"])

# Discriminative learning rates: the pretrained trunk moves slowly, the fresh
# head moves fast. A single LR either destroys the AudioSet features or leaves
# the head untrained.
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": CFG["lr_backbone"]},
    {"params": list(model.norm.parameters()) + list(model.head.parameters()),
     "lr": CFG["lr_head"]},
], weight_decay=CFG["weight_decay"])

steps_per_epoch = int(np.ceil(len(train_loader) / CFG["grad_accum_steps"]))
total_steps = steps_per_epoch * CFG["num_epochs"]
scheduler = get_cosine_schedule_with_warmup(
    optimizer, int(CFG["warmup_ratio"] * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"] and device.type == "cuda")
print(f"{total_steps} optimiser steps over {CFG['num_epochs']} epochs")

---
## 7. Training

Best epoch is selected on the **official ICBHI challenge score**,
`(Se + Sp) / 2` with abnormal classes pooled — not the macro variant in
`Model_Training_Protocol.md §3`, which is not comparable to any published ICBHI
number. Both are logged; see the note at the top of `owmtl/reporting.py`.

Checkpointing: full resume state to local disk every epoch, model weights to
Drive whenever the metric improves, and the training history to Drive every
epoch. History is the thing you cannot cheaply recompute after a disconnect.

In [ ]:
import time, json
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, loader, *, want_embeddings=False):
    model.eval()
    logits_all, labels_all, embs = [], [], []
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            out = model(xb, return_embedding=want_embeddings)
            logits, emb = out if want_embeddings else (out, None)
            loss = criterion(logits.float(), yb)
        total_loss += float(loss) * yb.size(0); n += yb.size(0)
        logits_all.append(logits.float().cpu()); labels_all.append(yb.cpu())
        if want_embeddings:
            embs.append(emb.float().cpu())
    logits = torch.cat(logits_all); labels = torch.cat(labels_all)
    metrics = reporting.compute_metrics(labels.numpy(), logits.argmax(1).numpy(),
                                        icbhi.SOUND_EVENT_CLASSES)
    metrics["loss"] = total_loss / max(n, 1)
    return metrics, (torch.cat(embs).numpy() if want_embeddings else None), logits.numpy()


def save_ckpt(path, epoch, best, history, with_optimizer=True):
    state = {"epoch": epoch, "best": best, "history": history,
             "model": model.state_dict(), "config": CFG}
    if with_optimizer:
        state |= {"optimizer": optimizer.state_dict(),
                  "scheduler": scheduler.state_dict(),
                  "scaler": scaler.state_dict()}
    torch.save(state, path)

In [ ]:
LATEST = os.path.join(CKPT_LOCAL, "latest.pt")
BEST_D = os.path.join(CKPT_DRIVE, "best_model.pth")
HIST_D = os.path.join(CKPT_DRIVE, "history.json")

start_epoch, best_score, history = 1, -1.0, []

# --- auto-resume: local full state first, else Drive weights -----------------
if os.path.exists(LATEST):
    ck = torch.load(LATEST, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, best_score, history = ck["epoch"] + 1, ck["best"], ck["history"]
    print(f"resumed from local epoch {ck['epoch']}")
elif os.path.exists(BEST_D) and os.path.exists(HIST_D):
    ck = torch.load(BEST_D, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    history = json.load(open(HIST_D))
    start_epoch = max(h["epoch"] for h in history) + 1
    best_score = ck["best"]
    print(f"resumed from Drive weights at epoch {start_epoch - 1} "
          f"(optimiser state rebuilt)")

epoch_times = []
for epoch in range(start_epoch, CFG["num_epochs"] + 1):
    model.train()
    t0 = time.time()
    run_loss, seen, correct = 0.0, 0, 0
    tr_logits, tr_labels = [], []
    optimizer.zero_grad(set_to_none=True)

    bar = tqdm(train_loader, desc=f"epoch {epoch}/{CFG['num_epochs']}", leave=False)
    for step, (xb, yb) in enumerate(bar):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            logits = model(xb)
            loss = criterion(logits.float(), yb) / CFG["grad_accum_steps"]
        scaler.scale(loss).backward()

        if (step + 1) % CFG["grad_accum_steps"] == 0 or step + 1 == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()

        run_loss += float(loss) * CFG["grad_accum_steps"] * yb.size(0); seen += yb.size(0)
        correct += int((logits.argmax(1) == yb).sum())
        tr_logits.append(logits.detach().float().argmax(1).cpu()); tr_labels.append(yb.cpu())
        bar.set_postfix(loss=f"{run_loss / seen:.4f}", acc=f"{correct / seen:.3f}")

    from sklearn.metrics import f1_score
    tr_f1 = f1_score(torch.cat(tr_labels).numpy(), torch.cat(tr_logits).numpy(),
                     average="macro", zero_division=0)
    val_metrics, _, _ = evaluate(model, val_loader)
    dt = time.time() - t0; epoch_times.append(dt)

    history.append({
        "epoch": epoch,
        "train_loss": run_loss / seen, "val_loss": val_metrics["loss"],
        "train_accuracy": correct / seen, "val_accuracy": val_metrics["accuracy"],
        "train_f1_macro": float(tr_f1), "val_f1_macro": val_metrics["f1_macro"],
        "val_icbhi_score": val_metrics["icbhi_score"],
        "lr": scheduler.get_last_lr()[0], "epoch_time_s": round(dt, 2),
    })

    print(f"epoch {epoch:>2}  train_loss {run_loss / seen:.4f}  "
          f"val_loss {val_metrics['loss']:.4f}  val_F1 {val_metrics['f1_macro']:.4f}  "
          f"val_ICBHI {val_metrics['icbhi_score']:.4f}  ({dt:.0f}s)")

    save_ckpt(LATEST, epoch, best_score, history)
    json.dump(history, open(HIST_D, "w"), indent=2)

    if val_metrics["icbhi_score"] > best_score:
        best_score = val_metrics["icbhi_score"]
        save_ckpt(BEST_D, epoch, best_score, history, with_optimizer=False)
        print(f"          new best ICBHI score {best_score:.4f} -> saved to Drive")

print(f"\ndone. best val ICBHI score {best_score:.4f}")

---
## 8. Test evaluation and the results JSON

Loads the best checkpoint, evaluates on the held-out test partition, and writes
the protocol-§4 artifacts: `results_<ID>.json`, the three curve plots, and the
confusion matrix.

In [ ]:
ck = torch.load(BEST_D, map_location=device, weights_only=False)
model.load_state_dict(ck["model"])
best_epoch = ck["epoch"]
print(f"loaded best checkpoint from epoch {best_epoch}")

test_metrics, test_emb, test_logits = evaluate(model, test_loader, want_embeddings=True)
test_metrics.pop("loss", None)

print(f"\nTEST — {MODEL_ID}  ({SCHEME} split)")
print(f"  accuracy          {test_metrics['accuracy']:.4f}")
print(f"  macro F1          {test_metrics['f1_macro']:.4f}")
print(f"  macro precision   {test_metrics['precision_macro']:.4f}")
print(f"  macro recall      {test_metrics['recall_macro']:.4f}")
print(f"  macro specificity {test_metrics['specificity_macro']:.4f}")
print(f"  ICBHI score       {test_metrics['icbhi_score']:.4f}"
      f"   (Se {test_metrics['icbhi_sensitivity']:.4f} / Sp {test_metrics['icbhi_specificity']:.4f})")
print(f"  ICBHI (macro var) {test_metrics['icbhi_score_macro']:.4f}")
print("\n  per class:")
for cls, m in test_metrics["per_class"].items():
    print(f"    {cls:<9} P {m['precision']:.3f}  R {m['recall']:.3f}  "
          f"F1 {m['f1']:.3f}  Sp {m['specificity']:.3f}  n={m['support']}")

In [ ]:
inference_ms = reporting.measure_inference_ms(model, test_ds[0][0].unsqueeze(0))

payload = reporting.build_results(
    model_id=MODEL_ID, model_name=MODEL_NAME, member=MEMBER, member_name=MEMBER_NAME,
    config=CFG | {"audio": CFG_AUDIO.to_dict()},
    dataset_info={
        "dataset": "ICBHI_2017",
        "task": "sound_event_4class",
        "train_samples": len(train_rows), "val_samples": len(val_rows),
        "test_samples": len(test_rows),
        "train_patients": len({r["patient_id"] for r in train_rows}),
        "test_patients": len({r["patient_id"] for r in test_rows}),
        "split_method": f"patient_independent_{SCHEME}_split_v1",
        "split_artifact": "splits/split_v1.json",
        "unknown_disease_patients_excluded_from_training": sorted(UNKNOWN_PIDS),
    },
    efficiency={
        **reporting.count_params(model),
        "model_size_mb": reporting.model_size_mb(model),
        "training_time_total_s": round(sum(h["epoch_time_s"] for h in history), 1),
        "training_time_per_epoch_s_avg": round(
            np.mean([h["epoch_time_s"] for h in history]), 1),
        "gpu_name": reporting.environment_block()["gpu_name"],
        "inference_time_ms_per_sample": inference_ms,
    },
    best_epoch={"epoch": best_epoch, "primary_metric": "icbhi_score",
                "primary_metric_value": round(best_score, 4)},
    best_metrics=test_metrics,
    ablation={
        "ablation_group": "augmentation_effect" if USE_SPECAUGMENT else "backbone_architecture",
        "ablation_role": "variant",
        "baseline_model_id": "M4" if USE_SPECAUGMENT else "M12",
        "variable_changed": ("augmentation: SpecAugment" if USE_SPECAUGMENT
                             else "backbone: AST_pretrained"),
        "variables_held_constant": [
            "loss_function: weighted CrossEntropyLoss", "optimizer: AdamW",
            f"data_split: {SCHEME}_split_v1", f"seed: {SEED}",
            "preprocessing: 128mel_16kHz_8s_kaldi_fbank",
            "augmentation: none" if not USE_SPECAUGMENT else "backbone: AST_pretrained",
        ],
        "component_flags": {
            "has_sound_event_head": True, "has_disease_head": False,
            "has_cross_task_consistency": False, "has_cqkd_regularization": False,
            "has_openmax_rejection": False, "owl_stage": 0, "compression_clusters": None,
        },
        "loss_weights": {"sound_event_weight": 1.0, "disease_weight": None,
                         "consistency_weight": None},
        "secondary_ablation_groups": ["augmentation_effect"] if not USE_SPECAUGMENT else [],
    },
    training_history=history,
    is_augmented=USE_SPECAUGMENT,
    augmentation_method=(json.dumps(features.spec_augment_config())
                         if USE_SPECAUGMENT else "none"),
    notes=("ICBHI score is the official challenge definition (Se/Sp with abnormal "
           "classes pooled); icbhi_score_macro is the Model_Training_Protocol §3 "
           "variant, retained for compatibility. AST uses its own Kaldi-fbank "
           "front-end to match its pretraining, not the shared librosa log-mel."),
)

out = reporting.finalise_run(payload, icbhi.SOUND_EVENT_CLASSES, RESULT_DIR)
print("results JSON :", out["results_json"])
for p in out["plots"]:
    print("plot         :", p)
print("\nprotocol check:", "PASSED" if not out["problems"] else out["problems"])

---
## 9. Open-world separability — the M12 selection criterion

This is the contribution. Everything above is a competent benchmark run; this
section is the measurement claim.

The backbone has been trained on **sound-event labels only**. It has never seen a
disease label. We now:

1. embed every cycle of the known-disease patients and of the 19 held-out
   unseen-disease patients;
2. mean-pool to one vector per patient;
3. fit an unknown-detector on the **known-disease training patients only**;
4. score known-test vs unseen-disease patients.

Nothing is fitted on the unknown group — it is scored exactly as Member B will
score it later. The question is whether the representation that maximises
sound-event F1 is also the one that best exposes unseen disease. If not, then
**accuracy is the wrong criterion for selecting the shared encoder**, and M12
should not be decided on the ablation table alone.

With n=19 unknown patients, report AUROC as a point estimate. No bootstrap CI —
the project's statistical protocol forbids it below n=15 and 19 is not meaningfully
above it.

In [ ]:
@torch.no_grad()
def embed(rows):
    ds = CycleDataset(rows, FEATS, UID_TO_ROW, augment=False)
    dl = DataLoader(ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=2)
    model.eval()
    out = []
    for xb, _ in tqdm(dl, desc="embedding", leave=False):
        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            _, emb = model(xb.to(device), return_embedding=True)
        out.append(emb.float().cpu().numpy())
    return np.concatenate(out), np.array([r["patient_id"] for r in rows])

PATIENT_DX = {int(p): v["diagnosis"] for p, v in SPLIT["patients"].items()}
known_train_pids = set(icbhi.disease_patients(SPLIT, "known_train"))
known_test_pids  = set(icbhi.disease_patients(SPLIT, "known_test"))

kt_rows = [r for r in INDEX if r["patient_id"] in known_train_pids]
ke_rows = [r for r in INDEX if r["patient_id"] in known_test_pids]

E_kt, P_kt = embed(kt_rows)
E_ke, P_ke = embed(ke_rows)
E_un, P_un = embed(held_rows)

X_kt, pid_kt = separability.aggregate_by_patient(E_kt, P_kt)
X_ke, pid_ke = separability.aggregate_by_patient(E_ke, P_ke)
X_un, pid_un = separability.aggregate_by_patient(E_un, P_un)
y_kt = [PATIENT_DX[p] for p in pid_kt]

print(f"known-train {X_kt.shape}  known-test {X_ke.shape}  unknown {X_un.shape}")

In [ ]:
sep = separability.open_world_separability(X_kt, y_kt, X_ke, X_un, n_components=32, seed=SEED)

print(f"OPEN-WORLD SEPARABILITY — {MODEL_ID}")
print(f"  scored {sep.n_known_test} known-disease vs {sep.n_unknown} unseen-disease patients")
print("\n  unknown-detection AUROC (higher = representation exposes unseen disease better)")
for k, v in sep.auroc.items():
    print(f"    {k:<14} {v:.4f}   (AUPRC {sep.auprc[k]:.4f})")
print(f"\n  Fisher ratio, known diseases : {sep.fisher_ratio_known:.4f}")
print(f"  silhouette, known vs unknown : {sep.silhouette:.4f}")
print("\n  0.5 AUROC means the representation carries no unseen-disease signal at all;")
print("  the sound-event F1 above tells you nothing about which way this went.")

sep_path = os.path.join(RESULT_DIR, f"separability_{MODEL_ID}.json")
json.dump({"model_id": MODEL_ID,
           "sound_event_f1_macro": test_metrics["f1_macro"],
           "sound_event_icbhi_score": test_metrics["icbhi_score"],
           "separability": sep.to_dict()},
          open(sep_path, "w"), indent=2)
print("\nwrote", sep_path)

In [ ]:
# --- 9.1 Figure: where do unseen diseases sit in the representation? ---------
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=SEED).fit(X_kt)
fig, ax = plt.subplots(figsize=(7, 5.5))
palette = {"COPD": "#2b6cb0", "Healthy": "#38a169", "URTI": "#d69e2e"}
for dx, colour in palette.items():
    m = np.array([PATIENT_DX[p] == dx for p in pid_ke])
    if m.any():
        Z = pca.transform(X_ke[m])
        ax.scatter(Z[:, 0], Z[:, 1], c=colour, s=55, label=f"{dx} (known)",
                   edgecolor="white", linewidth=0.6)
Z = pca.transform(X_un)
ax.scatter(Z[:, 0], Z[:, 1], marker="X", c="#c53030", s=110,
           label="unseen disease (n=19)", edgecolor="white", linewidth=0.8)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title(f"{MODEL_ID}: patient embeddings from a sound-event-only backbone")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.2, linewidth=0.5)
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, f"separability_pca_{MODEL_ID}.png"), dpi=150)
plt.show()

In [ ]:
# --- 9.2 The M12 table (run once every backbone's separability JSON exists) ---
import glob

files = sorted(glob.glob(os.path.join(DRIVE_ROOT or ".", "**", "separability_*.json"),
                         recursive=True))
if len(files) < 2:
    print(f"only {len(files)} backbone(s) measured so far — this cell becomes the M12 "
          f"decision table once M2, M3 and M4 have all been run.\n"
          f"Barshon's M2/M3 runs need Section 9 appended to produce their JSON.")
else:
    results = {}
    for f in files:
        d = json.load(open(f))
        results[d["model_id"]] = {"f1_macro": d["sound_event_f1_macro"],
                                  "separability": d["separability"]}
    verdict = separability.compare_backbones(results)
    print(f"{'backbone':<10}{'sound-event F1':>16}{'OW separability':>18}{'Fisher':>10}")
    for row in verdict["table"]:
        print(f"{row['backbone']:<10}{row['sound_event_f1_macro']:>16.4f}"
              f"{row['separability_auroc']:>18.4f}{row['fisher_ratio']:>10.3f}")
    print(f"\nbest by F1           : {verdict['winner_by_f1']}")
    print(f"best by separability : {verdict['winner_by_separability']}")
    print(f"criteria disagree    : {verdict['criteria_disagree']}")
    json.dump(verdict, open(os.path.join(DRIVE_ROOT or ".", "M12_backbone_selection.json"),
                            "w"), indent=2)

---
## 10. What goes on the slides (Aug 8)

Six minutes. One slide each:

1. **Task and data** — ICBHI 2017, 6,898 cycles, 126 patients, 4 sound-event
   classes. State the class imbalance (Normal 3,387 → Both 471) up front.
2. **Preprocessing** — the seven-stage list from Section 3 plus the pipeline
   figure. This is separately graded; read the list aloud.
3. **Split** — patient-independent, and the anti-leak rule. One sentence:
   *"the 19 unseen-disease patients are held out of training for every member, so
   the open-world result later cannot be memorisation."* This is the slide that
   distinguishes the group's work from the other teams'.
4. **Model** — AST, AudioSet-pretrained, 86.6 M params, why a transformer over a
   CNN for spectrograms, discriminative learning rates.
5. **Results** — clean vs SpecAugment side by side: accuracy, macro P/R/F1,
   specificity, ICBHI score, confusion matrix, params, size, train time,
   inference latency.
6. **Beyond the assignment** — the separability figure from Section 9 and the
   one-line claim: *accuracy is not the right criterion for selecting a shared
   encoder in an open-world pipeline, and here is the measurement.*

### Before you call this done

- [ ] Both runs finished (`USE_SPECAUGMENT` False **and** True)
- [ ] `results_M4.json` and `results_M23.json` in Drive, `problems` empty
- [ ] Four plots per run: loss, accuracy, F1, confusion matrix
- [ ] Params / size / train time / inference latency recorded
- [ ] `separability_M4.json` written
- [ ] Preprocessing stages listed explicitly on a slide, not just in code
- [ ] Committed `splits/` artifacts pushed so B, C and D can start